multihead, multiblocks

In [1]:
from gpt_data import *

In [11]:
class AttentionHead(nn.Module):
	def __init__(self, C, d):
		super().__init__()
		self.B, self.T, self.C, self.d = batchsize, context_size, C, d
		self.Q, self.K, self.V = nn.Linear(self.C, self.d), nn.Linear(self.C, self.d), nn.Linear(self.C, self.d)
		self.register_buffer("mask", torch.tril(torch.ones(self.T, self.T, dtype=torch.bool))) # move with the weights

	def forward(self, x):
		q, k, v = self.Q(x), self.K(x), self.V(x)
		scores = q @ k.mT / self.d**0.5
		scores = scores.masked_fill(~self.mask, float('-inf'))
		att = torch.softmax(scores, dim=-1)
		return att @ v

class Block(nn.Module):
	def __init__(self, C, d, nheads):
		super().__init__()

		self.C, self.d = C, d

		self.heads = nn.ModuleList([AttentionHead(self.C, self.d) for _ in range(nheads)])

		self.W0 = nn.Linear(self.C, self.C)

		self.mlp = nn.Sequential(
			nn.Linear(self.C, self.C*4),
			nn.ReLU(),
			nn.Linear(self.C*4, self.C),
			nn.Dropout(0.1)
		)
		self.ln1 = nn.LayerNorm(self.C)
		self.ln2 = nn.LayerNorm(self.C)

	def forward(self, x):
		h = self.ln1(x)
		all_heads = torch.cat([attn(h) for attn in self.heads], dim=-1)
		x = x + self.W0(all_heads)
		x = x + self.mlp(self.ln2(x))
		return x


class MultiAttention(nn.Module):
	def __init__(self, embedding_size=64, nheads=4, nblocks=4):
		super().__init__()

		self.B, self.T, self.C = batchsize, context_size, embedding_size
		self.d = self.C // nheads

		self.embeddings = nn.Embedding(27, self.C)
		self.pos = nn.Embedding(self.T, self.C) 

		self.blocks = nn.ModuleList([Block(self.C, self.d, nheads) for _ in range(nblocks)])

		self.ln_f = nn.LayerNorm(self.C)
		self.head = nn.Linear(self.C, 27)
	
	def forward(self, tokens):
		tok_emb, pos_emb = self.embeddings(tokens), self.pos(torch.arange(self.T))
		x = tok_emb + pos_emb            # (B, T, C)

		for block in self.blocks:
			x = block(x)

		logits = self.head(self.ln_f(x)) 
		return logits

mymodel = MultiAttention()

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mymodel.parameters()) 

lossi = []

for i in range(10000):
	ixs = torch.randint(0, Xtrain.shape[0], (batchsize,))
	ctx, target = Xtrain[ixs], Ytrain[ixs]

	logits = mymodel(ctx)

	loss = criterion(logits.mT, target)

	optimizer.zero_grad()
	loss.backward()
	optimizer.step()

	lossi.append(loss.item())

print(loss.item())

1.1210992336273193


In [13]:
alpha = "abcdefghijklmnopqrstuvwxyz."

mymodel.eval() 

with torch.no_grad(): 
	for i in range(20):
		ctx = torch.tensor([26] * context_size).unsqueeze(0)
		name = ""

		while len(name) < 20:
			logits = mymodel(ctx)
			probs = torch.softmax(logits[:, -1, :], axis=1)
			cidx = torch.multinomial(probs, 1)
			c = alpha[cidx.item()]; name += c
			if c == ".":
				break
			ctx = torch.cat([ctx[:, 1:], cidx], dim=1)
		
		print(name)

ammy.
velina.
azenda.
madisyn.
ditie.
alissa.
jazzlin.
daliya.
pamelina.
finlee.
daviya.
kaileigh.
zariyah.
juliette.
analice.
flor.
raya.
paysl.
mason.
karlene.


In [14]:
@torch.no_grad()
def get_loss(split):
	X, Y = {
		"train": (Xtrain, Ytrain),
		"val": (Xval, Yval)
	}[split]

	ixs = torch.randint(0, X.shape[0], (batchsize,))
	ctx, target = X[ixs], Y[ixs]

	logits = mymodel(ctx)
	loss = F.cross_entropy(logits.view(-1, 27), target.view(-1))

	print(split, loss.item())

get_loss("train")
get_loss("val")

train 1.0529148578643799
val 1.6706323623657227
